# Generation for Charging Stations depending on relative Path Occupation

## Research Question

The feasibility of intelligently placing electrical charging stations on highly visited roads. SUMO and MatSIM deliver a full framework for simulating realistic traffic.


## Methods to determine traffic

- Occupation of lanes
- Color sections by own criteria, depending on traffic (manually, or by automation script) 
- Occupation of certain areas

### Calculate with Occupation
ex: Lane A is occupied by 2 cars, X with 5s (seconds) and Y with 10s, in a timeframe of 60s. Car X enters on second 20-25, car Y enters on 23-33.
The occupancy of Lane A is a total of 13 seconds of 60. 
$$ ( 13 / 60 = 0,2167 ) $$

Mathematically, the lane is occupied 21.67% of the time.  

### Calculate with Density    
ex: Lane A (500m) is populated with 10 cars.
$$ 10 (cars) / 0.5 (km) =  20  cars / km $$ 

With lots of traffic the value may approach a value of around 100 cars/km.

## Chosen method to determine traffic

The problem requires a different solution than the occupation of a lane, because a high occupation does not necessarily hint towards a lot of traffic. So, the initial method will be utilizing the density of each lane across the whole simulation time span.

Density allows to track every lane independently and extract information for these lanes. For now Occupation is less of a prioritiy.

## Design of method

The first step to complete the goal of finding a way to calculate beneficial positions for charging stations, is to extract the density and the lane's id, to which it belongs. These two values do not solve the problem itself, but deliver the information needed for setting up the criteria and a fitting algorithm.  


## Implementation of method

Below is the first part of code, that allows the extraction of the density and lane id into two variables.

In [ ]:
file_name = 'osm'

In [ ]:
import xml.etree.ElementTree as ET

tree = ET.parse(file_name + '.xml')
root = tree.getroot()

lane_data = []
for edge in root.findall('.//edge'):
    for lane in edge.findall('.//lane'):
        density = edge.get('density')
        id = edge.get('id')
        lane_data.append(lane.attrib)

#print(lanes)


## Algorithm to determine positioning of charging stations

The algorithm will take into account which lanes are busy and will put electrical charging stations depending on their traffic density.

## Requirements Engineering

It does not make sense to put charging stations everywhere, where there is a density value higher than X, assuming some streets might be overcrowded and others void of charging stations. So, a few criterias need to be elaborated to guarantee and efficient system to distribute charging stations.

## Modify sumocfg file

For the sumocfg to be able to handle additional information, a tag with "additional-files" needs to be added.

In [ ]:
import os

def get_addition_files():
    add_files = []
    for file in os.listdir():
        if file.endswith('add.xml'):
            add_files.append(file)
    return add_files


sumo_cfg = file_name + '.sumocfg'
sumo_network_cfg = ET.parse(sumo_cfg)
sumo_root_cfg = sumo_network_cfg.getroot()


inserted_adds = []
for additionals in sumo_network_cfg.findall('.//additional-files'):
    inserted_adds.append(list(additionals.attrib.values())[0])


#Only add "additional-files" with value -lanes.add.xml
#Maximum of one additional-files allowed in sumocfg
add = get_addition_files()
for name in add: 
    if name in inserted_adds:
        print(f"additional-file %s already exists" % name)
    else:
        additional = ET.Element('additional-files')
        additional.set('value', name)
        if ('lane' in name):
            sumo_root_cfg.append(additional)


sumo_network_cfg.write(sumo_cfg, encoding='utf-8', xml_declaration=True)


## Create the additional-files XML

The essential file for any further additions, like charging stations, adjustments for cars etc.

In [ ]:
destination_lanes_add = file_name + ".add.xml"

if not os.path.exists(destination_lanes_add):
    xml_content = """<?xml version="1.0" encoding="UTF-8"?>\n<additional>\n</additional>"""
    with open(destination_lanes_add, "w", encoding="utf-8") as file:
        file.write(xml_content)

## Retrieve lenghts of network lanes to their lanes.add

Sustaining the retrieved data will be useful later on. For now, only the length value has been added to the lane_data dictionary, so that lanes, which are too short, are not considered for having any charging stations.

In [ ]:
network_cfg = file_name + ".net.xml"
network_tree = ET.parse(network_cfg)
network_root = network_tree.getroot()

#network_lane_data consists of all lanes (that are not internal) and their information
network_lane_data = []
for edge in network_root.findall('.//edge'):
    if edge.attrib.get('function') == 'internal':
        continue

    for lane in edge.findall('.//lane'):
        network_lane_data.append(lane.attrib)
        #print(list(edge.attrib.values())[1])

#print(network_lane_data[1])

for lane_id in lane_data:
    #print(lane_id.get('id'))
    for network_lane_id in network_lane_data:
        if lane_id.get('id') == network_lane_id.get('id'):
            print('')
            lane_id['length'] = network_lane_id.get('length')
#print(lane_data)

## Algorithm to add charging stations

In [ ]:
network_lane_ids = []

#deprecated - was necessary with old lanes.xml
def retrieve_real_lane_ids():
    for i in range(len(lane_data)):
        modified = False
        if (lane_data[i-1]):
            if (lane_data[i] != lane_data[i-1]):
                modified = True
                lane_data[i] = str(lane_data[i]) + "_0"
        if not modified:
            lane_data[i] = str(lane_data[i]) + "_1"

    network_lane_ids = lane_data

def create_charging_stations(add_file):
    additional_tree = ET.parse(add_file)
    additional_root = additional_tree.getroot()

    for cs in additional_root.findall('.//chargingStation'):
        additional_root.remove(cs)

    for data in lane_data:
        current_lane_id = data.get('id')
        # add: """& float(data.get('density')) > 4""" if addressing for density needed
        if(float(data.get('length')) > 20 ):
            cs = ET.SubElement(additional_root, 'chargingStation')
            cs.set('id', 'cs_'+current_lane_id)
            cs.set('lane', current_lane_id)
            cs.set('startPos', str(float(data.get('length')) * 1/4))
            cs.set('endPos', str(float(data.get('length')) * 2/4))
            #print("::" + str(cs))

    additional_tree.write(file_name + ".add.xml", encoding='utf-8', xml_declaration=True)



create_charging_stations(destination_lanes_add)


### Considering different criteria

Traffic can be looked at through basic values like density or occupancy, while others can still deliver some valuable information the task of smart CS-distribution.

    Average number of vehicles on the edge (#) = sampledSeconds /  period
    Average traffic volume (#/h) = speed * 3.6 * density
    Traffic volume at the begin of the lane / edge (#/h) = 3600 * entered / period
    Traffic volume at the end of the lane / edge (#/h) = 3600 * left /  period
    Total distance travelled (m) = speed * sampledSeconds
    Edge length = sampledSeconds / period * 1000 / density
